<a href="https://colab.research.google.com/github/Mythri-S24/HealthChatBot/blob/main/genai_openended.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-learn numpy requests

In [ ]:
!pip install transformers torch sentencepiece -q

In [ ]:
!pip install gradio

In [ ]:
# ============================================================
# Domain-Specific Chatbot with RAG
# Domain: Healthcare FAQ
# Open-Source LLM Version (FLAN-T5)
# ============================================================

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# -------------------------------------------------------
# 1. HARDCODED KNOWLEDGE BASE (Healthcare FAQs)
# -------------------------------------------------------
KNOWLEDGE_BASE = [
    {"id": 1, "topic": "Fever",
     "content": "A fever is a temporary rise in body temperature. Normal body temperature is 98.6F (37C). A temperature above 100.4F (38C) is considered a fever. Common causes include infections, vaccinations, and inflammatory conditions. Adults with fever above 103F should seek medical attention."},

    {"id": 2, "topic": "Diabetes",
     "content": "Diabetes is a chronic disease affecting how the body processes blood sugar. Type 1 is an autoimmune condition requiring insulin. Type 2 is often lifestyle-related and managed through diet, exercise, and medication. Symptoms include frequent urination, excessive thirst, and fatigue."},

    {"id": 3, "topic": "Hypertension",
     "content": "Hypertension (high blood pressure) is when blood pressure readings consistently exceed 130/80 mmHg. It is a major risk factor for heart disease and stroke. Management includes dietary changes (low salt), exercise, stress reduction, and antihypertensive medications."},

    {"id": 4, "topic": "COVID-19",
     "content": "COVID-19 is caused by the SARS-CoV-2 virus. Symptoms include fever, cough, fatigue, and loss of taste or smell. Vaccination is the primary prevention method. Treatment is mostly supportive; antivirals may be prescribed in high-risk cases."},

    {"id": 5, "topic": "Mental Health",
     "content": "Mental health includes emotional, psychological, and social well-being. Common disorders include depression, anxiety, and bipolar disorder. Treatments involve therapy (CBT), medications (antidepressants), and lifestyle changes. Early intervention improves outcomes significantly."},

    {"id": 6, "topic": "Nutrition",
     "content": "A balanced diet includes carbohydrates, proteins, healthy fats, vitamins, and minerals. WHO recommends at least 5 portions of fruits and vegetables per day."},

    {"id": 7, "topic": "Exercise",
     "content": "WHO recommends 150-300 minutes of moderate aerobic activity weekly for adults. Exercise reduces cardiovascular disease and improves mental health."},

    {"id": 8, "topic": "First Aid - Burns",
     "content": "For minor burns, cool under running water for 10-20 minutes. Avoid ice, butter, or toothpaste. Cover with sterile bandage."},
]

# -------------------------------------------------------
# 2. RETRIEVER MODULE (TF-IDF + Cosine Similarity)
# -------------------------------------------------------
class DocumentRetriever:
    """Retrieves relevant documents using TF-IDF."""

    def __init__(self, knowledge_base, num_docs=2):

        self.knowledge_base = knowledge_base
        self.num_docs = num_docs
        self.texts = [doc['content'] for doc in knowledge_base]

        self.vectorizer = TfidfVectorizer(
            stop_words='english',
            ngram_range=(1, 2)
        )

        self.tfidf_matrix = self.vectorizer.fit_transform(
            self.texts
        )

        print(f'[Retriever] Indexed {len(self.texts)} documents.')

    def retrieve(self, query):

        query_vec = self.vectorizer.transform([query])

        scores = cosine_similarity(
            query_vec,
            self.tfidf_matrix
        ).flatten()

        top_indices = np.argsort(scores)[::-1][:self.num_docs]

        results = []

        for idx in top_indices:
            if scores[idx] > 0:
                results.append({
                    'topic': self.knowledge_base[idx]['topic'],
                    'content': self.knowledge_base[idx]['content'],
                    'score': round(float(scores[idx]), 4)
                })

        return results


# -------------------------------------------------------
# 3. OPEN-SOURCE LLM (FLAN-T5)
# -------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


# -------------------------------------------------------
# 3. OPEN-SOURCE LLM (FLAN-T5)
# -------------------------------------------------------
class LLMGenerator:
    """Generates responses using FLAN-T5."""

    def __init__(self,
                 temperature=0.7,
                 max_tokens=150):

        self.temperature = temperature
        self.max_tokens = max_tokens

        print("[LLM] Loading FLAN-T5 model...")

        self.model_name = "google/flan-t5-base"

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )

        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name
        )

        print("[LLM] Model loaded successfully.")

    def generate(self, query, context_docs):

        if not context_docs:
             return (
            "I could not find relevant "
            "information for your query."
            )

    # Use only best retrieved document
        best_doc = context_docs[0]['content']

        prompt = (
        f"You are a healthcare assistant.\n"
        f"Answer ONLY from the information below.\n\n"
        f"Information:\n{best_doc}\n\n"
        f"Question: {query}\n\n"
        f"Give a clear answer in 2-3 sentences."
        )

        inputs = self.tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
        )

        outputs = self.model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        num_beams=5,
        early_stopping=True
        )

        response = self.tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
        )

        return response

# -------------------------------------------------------
# 4. CHATBOT PIPELINE
# -------------------------------------------------------
class HealthcareChatbot:
    """End-to-end RAG chatbot."""

    def __init__(self,
                 num_docs=2,
                 temperature=0.7,
                 max_tokens=150):

        print('Initializing Healthcare Chatbot...')

        self.retriever = DocumentRetriever(
            KNOWLEDGE_BASE,
            num_docs=num_docs
        )

        self.generator = LLMGenerator(
            temperature=temperature,
            max_tokens=max_tokens
        )

        print('Chatbot ready.\n')

    def chat(self, query):

        print(f'User: {query}')

        retrieved = self.retriever.retrieve(query)

        print(
            f'[Retrieved {len(retrieved)} doc(s)]:',
            [d['topic'] for d in retrieved]
        )

        response = self.generator.generate(
            query,
            retrieved
        )

        print(f'Bot: {response}\n')

        return response


# -------------------------------------------------------
# 5. MAIN - INTERACTIVE SESSION
# -------------------------------------------------------
if __name__ == '__main__':

    bot = HealthcareChatbot(
        num_docs=2,
        temperature=0.7,
        max_tokens=150
    )

    print('Healthcare Chatbot (type "quit" to exit)')
    print('-' * 50)

    while True:

        user_input = input('You: ').strip()

        if user_input.lower() in [
            'quit', 'exit', 'bye'
        ]:
            print('Goodbye!')
            break

        if user_input:
            bot.chat(user_input)

Initializing Healthcare Chatbot...
[Retriever] Indexed 8 documents.
[LLM] Loading FLAN-T5 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[LLM] Model loaded successfully.
Chatbot ready.

Healthcare Chatbot (type "quit" to exit)
--------------------------------------------------
You: what are symptoms of fever
User: what are symptoms of fever
[Retrieved 2 doc(s)]: ['Fever', 'COVID-19']
Bot: A temperature above 100.4F

You: oexit
User: oexit
[Retrieved 0 doc(s)]: []
Bot: I could not find relevant information for your query.

You: exit
Goodbye!


In [ ]:
import gradio as gr

# ==========================================
# CONNECT BACKEND
# (Your backend must already be run)
# ==========================================
bot = HealthcareChatbot()


# ==========================================
# CHAT FUNCTION
# ==========================================
def chat_fn(message, history):

    if not message:
        return "", history

    history.append((message, "⏳ Thinking..."))

    response = bot.chat(message)

    history[-1] = (message, response)

    return "", history


# ==========================================
# FRONTEND UI
# ==========================================
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="MediCare AI"
) as demo:

    # ======================================
    # LANDING PAGE
    # ======================================
    with gr.Group(visible=True) as landing:

        gr.HTML("""
        <div style="
            height:100vh;
            background:
            linear-gradient(
            rgba(0,40,80,0.55),
            rgba(0,40,80,0.55)),
            url('https://images.unsplash.com/photo-1576091160399-112ba8d25d1d?auto=format&fit=crop&w=1600&q=80');

            background-size:cover;
            background-position:center;
            display:flex;
            justify-content:center;
            align-items:center;
        ">

            <div style="
                background:rgba(255,255,255,0.15);
                backdrop-filter: blur(18px);
                padding:60px;
                border-radius:30px;
                text-align:center;
                width:70%;
                max-width:750px;
                box-shadow:0px 10px 40px rgba(0,0,0,0.35);
                border:1px solid rgba(255,255,255,0.2);
            ">

                <h1 style="
                    color:white;
                    font-size:58px;
                    font-weight:800;
                    margin-bottom:10px;
                ">
                    🏥 MediCare AI
                </h1>

                <h2 style="
                    color:#E0F2FE;
                    font-size:28px;
                    margin-bottom:20px;
                ">
                    Smart Healthcare Assistant
                </h2>

                <p style="
                    color:white;
                    font-size:20px;
                    line-height:1.8;
                ">
                    AI-powered symptom analysis,
                    emergency detection,
                    and personalized healthcare guidance.
                </p>

                <br>

                <div style="
                    display:flex;
                    justify-content:center;
                    gap:20px;
                    flex-wrap:wrap;
                    color:white;
                    font-size:18px;
                    font-weight:600;
                ">
                    <span>✔ AI Diagnosis</span>
                    <span>✔ Symptom Checker</span>
                    <span>✔ Emergency Alerts</span>
                </div>

            </div>
        </div>
        """)

        start_btn = gr.Button(
            "Enter Healthcare System 🚀",
            variant="primary",
            size="lg"
        )

    # ======================================
    # CHATBOT PAGE
    # ======================================
    with gr.Group(visible=False) as chat:

        # ===== DASHBOARD HERO =====
        gr.HTML("""
        <div style="
            min-height:260px;
            background:
            linear-gradient(
            rgba(15,23,42,0.72),
            rgba(30,58,138,0.72)),
            url('https://images.unsplash.com/photo-1584982751601-97dcc096659c?auto=format&fit=crop&w=1600&q=80');

            background-size:cover;
            background-position:center;
            border-radius:28px;
            padding:50px;
            margin-bottom:25px;
            box-shadow:0px 10px 30px rgba(0,0,0,0.25);
        ">

            <h1 style="
                color:white;
                font-size:48px;
                font-weight:800;
                margin-bottom:12px;
            ">
                🏥 Welcome to MediCare AI
            </h1>

            <p style="
                color:#E2E8F0;
                font-size:20px;
                line-height:1.8;
                max-width:800px;
            ">
                Get AI-powered healthcare guidance,
                symptom analysis, emergency awareness,
                and medical information — all in one place.
            </p>

            <div style="
                display:flex;
                gap:18px;
                margin-top:25px;
                flex-wrap:wrap;
            ">

                <div style="
                    background:rgba(255,255,255,0.15);
                    padding:18px 28px;
                    border-radius:16px;
                    color:white;
                    backdrop-filter:blur(10px);
                ">
                    ✔ Symptom Checker
                </div>

                <div style="
                    background:rgba(255,255,255,0.15);
                    padding:18px 28px;
                    border-radius:16px;
                    color:white;
                    backdrop-filter:blur(10px);
                ">
                    ✔ Emergency Detection
                </div>

                <div style="
                    background:rgba(255,255,255,0.15);
                    padding:18px 28px;
                    border-radius:16px;
                    color:white;
                    backdrop-filter:blur(10px);
                ">
                    ✔ Medical Guidance
                </div>

            </div>
        </div>
        """)

        # ===== CHAT SECTION =====
        with gr.Row():

            # LEFT SIDE
            with gr.Column(scale=3):

                with gr.Group():

                    chatbot = gr.Chatbot(
                        height=520
                    )

                    msg = gr.Textbox(
                        label="Ask Your Health Question",
                        placeholder="Example: I have fever and headache..."
                    )

                    with gr.Row():

                        send = gr.Button(
                            "Send 🚀",
                            variant="primary"
                        )

                        clear = gr.Button(
                            "Clear 🧹"
                        )

                        back = gr.Button(
                            "⬅ Back"
                        )

            # RIGHT SIDE
            with gr.Column(scale=1):

                gr.HTML("""
                <div style="
                    background:rgba(255,255,255,0.14);
                    backdrop-filter:blur(16px);
                    padding:28px;
                    border-radius:22px;
                    color:white;
                    box-shadow:0px 8px 25px rgba(0,0,0,0.25);
                ">

                    <h2 style="color:white;">
                        🏥 Health Assistant
                    </h2>

                    <hr style="opacity:0.3">

                    <p>✔ Symptom Analysis</p>
                    <p>✔ Disease Guidance</p>
                    <p>✔ Emergency Detection</p>
                    <p>✔ Medical Information</p>

                    <br>

                    <p style="
                        font-size:13px;
                        opacity:0.85;
                    ">
                        ⚠ This AI provides guidance only.
                        Consult a doctor for emergencies.
                    </p>

                </div>
                """)

    # ======================================
    # NAVIGATION
    # ======================================
    def go_chat():
        return (
            gr.update(visible=False),
            gr.update(visible=True)
        )

    def go_home():
        return (
            gr.update(visible=True),
            gr.update(visible=False)
        )

    start_btn.click(
        go_chat,
        outputs=[landing, chat]
    )

    back.click(
        go_home,
        outputs=[landing, chat]
    )

    # ======================================
    # CHAT ACTIONS
    # ======================================
    send.click(
        chat_fn,
        [msg, chatbot],
        [msg, chatbot]
    )

    msg.submit(
        chat_fn,
        [msg, chatbot],
        [msg, chatbot]
    )

    clear.click(
        lambda: [],
        None,
        chatbot
    )


# ==========================================
# RUN APP
# ==========================================
demo.launch()

Initializing Healthcare Chatbot...
[Retriever] Indexed 8 documents.
[LLM] Loading FLAN-T5 model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[LLM] Model loaded successfully.
Chatbot ready.



/tmp/ipykernel_14418/676670005.py:30: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_14418/676670005.py:213: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_14418/676670005.py:213: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a18848970a89bff37d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import files
files.download('genai openended.ipynb')

FileNotFoundError: Cannot find file: genai openended.ipynb

In [ ]:
import os
os.listdir()

['.config', '.gradio', 'sample_data']